# 03 — Feature Engineering
**Zomato Project · Phase 3**

**Source Dataset:** Canonical cleaned output from `02_Data_Cleaning.ipynb`

> **Design Philosophy:** `02_Data_Cleaning.ipynb` intentionally preserves restaurants with missing ratings because they remain useful for the Flask application, NLP pipeline, recommendation engine, and RAG system. **This notebook does not clean data again.** Its sole responsibility is to create model-ready features and prepare separate datasets for supervised and non-supervised tasks. The distinction between cleaning and feature engineering is maintained as explicit, separate pipeline phases — cleaning ends at `02`, feature engineering begins here.

**Pipeline position:**
```
01_Data_Profiling → 02_Data_Cleaning → [03_Feature_Engineering] → 04_DecisionTree / 05_LightGBM / 06_SVM
```

**Scope of this notebook:**
- Target variable preparation (regression + classification)
- Business-oriented feature creation
- Categorical encoding (Binary / Label / One-Hot)
- Numerical feature preparation (log transform only — median imputation was completed in Phase 2)
- Feature scaling — SVM only
- NLP dataset preservation
- Train / Test split
- Assertions + Feature Engineering Summary + Export

**Reproducibility:** This notebook is deterministic. All splits use `random_state=42`. Running it multiple times on the same cleaned dataset will always produce identical outputs.

## 0 · Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_PATH   = Path('/Users/huntstar/Projects/Zomato_project/Data/zomato_cleaned_v1.csv')
OUTPUT_DIR   = Path('/Users/huntstar/Projects/Zomato_project/Data/')
RANDOM_STATE = 42

print('Libraries loaded.')
print(f'Input  → {INPUT_PATH}')
print(f'Output → {OUTPUT_DIR}')

Libraries loaded.
Input  → /Users/huntstar/Projects/Zomato_project/Data/zomato_cleaned_v1.csv
Output → /Users/huntstar/Projects/Zomato_project/Data


## 1 · Load Cleaned Dataset

In [2]:
df_clean = pd.read_csv(INPUT_PATH)
df       = df_clean.copy()   # ALL mutations happen on df — df_clean is never touched

print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print('\nDtypes:')
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Shape  : (51717, 14)
Columns: ['name', 'online_order', 'book_table', 'rate', 'votes', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_cost(for two people)', 'reviews_list', 'menu_item', 'listed_in(type)', 'listed_in(city)']

Dtypes:
name                               str
online_order                       str
book_table                         str
rate                           float64
votes                            int64
location                           str
rest_type                          str
dish_liked                         str
cuisines                           str
approx_cost(for two people)    float64
reviews_list                       str
menu_item                          str
listed_in(type)                    str
listed_in(city)                    str
dtype: object

Null counts:
rate          10052
dish_liked    28078
dtype: int64


## 2 · Feature Selection Strategy

**Why this step exists:**  
Not every column belongs in every model. Applying the same transformation pipeline to all columns wastes compute, inflates feature space, and risks leaking information across pipelines. This section explicitly maps each column to its role so that every downstream transformation targets only the columns that need it.

> **Important distinction:** Features such as `review_count`, `dish_count`, `cuisine_count`, and `cost_category` are generated for the **entire dataset** — not just supervised rows — because they are useful for recommendation, NLP, RAG, and the Flask application. Supervised filtering (excluding rows without a valid `rate`) does not happen until Section 10, which is **the first point in the pipeline where unrated restaurants are excluded**.

| Column | Type | Role | Pipeline |
|--------|------|------|----------|
| `votes` | Numerical | Feature | Tree + SVM |
| `approx_cost(for two people)` | Numerical | Feature | Tree + SVM |
| `online_order` | Binary | Encode → 0/1 | Tree + SVM |
| `book_table` | Binary | Encode → 0/1 | Tree + SVM |
| `location` | Categorical (93) | Label (Tree) / OHE (SVM) | Tree + SVM |
| `rest_type` | Categorical (93) | Label (Tree) / OHE (SVM) | Tree + SVM |
| `listed_in(type)` | Categorical (7) | Label Encode | Tree + SVM |
| `listed_in(city)` | Categorical (30) | Label (Tree) / OHE (SVM) | Tree + SVM |
| `cuisines` | Categorical (multi-value) | Cuisine Count feature | All datasets |
| `rate` | Float | Regression target | Regression |
| `rating_category` | String | Classification target | Classification |
| `reviews_list` | Text | Review Count feature + NLP | NLP |
| `dish_liked` | Text | Dish Count feature + NLP | NLP |
| `menu_item` | Text | RAG pipeline | RAG |
| `name` | Identifier | Preserved — not used as feature | Reference |

In [3]:
# ── Column role registry ──────────────────────────────────────────────────
NUMERICAL_COLS   = ['votes', 'approx_cost(for two people)']
BINARY_COLS      = ['online_order', 'book_table']
LABEL_ENC_COLS   = ['listed_in(type)', 'listed_in(city)', 'location', 'rest_type']
OHE_COLS         = ['location', 'rest_type', 'listed_in(city)']   # SVM only
TEXT_COLS        = ['reviews_list', 'dish_liked', 'menu_item']
TARGET_REG       = 'rate'
TARGET_CLF       = 'rating_category'
IDENTIFIER_COLS  = ['name']

print('Feature selection map registered.')
print(f'  Numerical  : {NUMERICAL_COLS}')
print(f'  Binary     : {BINARY_COLS}')
print(f'  Label Enc  : {LABEL_ENC_COLS}')
print(f'  OHE (SVM)  : {OHE_COLS}')
print(f'  Text       : {TEXT_COLS}')

Feature selection map registered.
  Numerical  : ['votes', 'approx_cost(for two people)']
  Binary     : ['online_order', 'book_table']
  Label Enc  : ['listed_in(type)', 'listed_in(city)', 'location', 'rest_type']
  OHE (SVM)  : ['location', 'rest_type', 'listed_in(city)']
  Text       : ['reviews_list', 'dish_liked', 'menu_item']


## 3 · Target Variable Preparation

**Why this step exists:**  
The `rate` column supports two different learning objectives. For regression, the raw float is the prediction target. For classification, the float must be converted into discrete quality categories so that the model learns to distinguish restaurant tiers rather than predict an exact decimal. Creating both representations here ensures that the same feature engineering pipeline feeds both experiment types without maintaining separate preprocessing chains.

| Rating Range | Category | Rationale |
|-------------|----------|-----------|
| 0.0 – 2.5 | Poor | Below half of max scale |
| 2.5 – 3.5 | Average | Middle band |
| 3.5 – 4.2 | Good | Above average |
| 4.2 – 5.0 | Excellent | Top tier |

In [4]:
# ── Regression target — already float, just confirm ───────────────────────
print(f'rate dtype  : {df["rate"].dtype}')
print(f'rate nulls  : {df["rate"].isnull().sum():,}  (retained — excluded at train time)')
print(f'rate range  : [{df["rate"].min():.1f}, {df["rate"].max():.1f}]')

# ── Classification target — bin into quality categories ──────────────────
RATING_BINS   = [0.0, 2.5, 3.5, 4.2, 5.0]
RATING_LABELS = ['Poor', 'Average', 'Good', 'Excellent']

df['rating_category'] = pd.cut(
    df['rate'],
    bins   = RATING_BINS,
    labels = RATING_LABELS,
    include_lowest = True
)

print('\nrating_category distribution:')
print(df['rating_category'].value_counts(dropna=False).sort_index())
print(f'\nNaN rating_category (from missing rate): {df["rating_category"].isnull().sum():,}')

rate dtype  : float64
rate nulls  : 10,052  (retained — excluded at train time)
rate range  : [1.8, 4.9]

rating_category distribution:
rating_category
Poor           288
Average      13996
Good         23297
Excellent     4084
NaN          10052
Name: count, dtype: int64

NaN rating_category (from missing rate): 10,052


## 4 · Business-Oriented Feature Engineering

**Why this step exists:**  
Raw columns like `votes` and `rate` carry useful signal but do not capture composite business concepts. Engineered features encode domain knowledge directly into the model — a restaurant with 4.8 stars and 12 votes is very different from one with 4.5 stars and 3,500 votes, but raw columns alone cannot express that distinction. Each feature below is designed to give tree-based and SVM models richer, more interpretable signal than the originals provide alone.

> All features in this section are created on the **full dataset** (including rows with missing `rate`) because they are useful across all downstream pipelines — not just supervised ML.

### 4.1 · Cost Category

Group restaurants into spending tiers based on approximate dining cost for two.

> `approx_cost(for two people)` was converted to numeric and imputed with the column median during Phase 2 (`02_Data_Cleaning.ipynb`). No imputation is needed here — `cost_category` is created once and does not need to be rebuilt.

In [5]:
COST_BINS   = [0, 300, 600, 800, float('inf')]
COST_LABELS = ['Budget', 'Mid-Range', 'Premium', 'Luxury']

df['cost_category'] = pd.cut(
    df['approx_cost(for two people)'],
    bins   = COST_BINS,
    labels = COST_LABELS,
    include_lowest = True
)

print('cost_category distribution:')
print(df['cost_category'].value_counts(dropna=False))
print(f'\nNaN cost_category: {df["cost_category"].isnull().sum():,}  (expected: 0 — approx_cost imputed in Phase 2)')

cost_category distribution:
cost_category
Mid-Range    19551
Budget       18554
Luxury        7845
Premium       5767
Name: count, dtype: int64

NaN cost_category: 0  (expected: 0 — approx_cost imputed in Phase 2)


### 4.2 · Cuisine Count
Number of distinct cuisine types served by a restaurant. A restaurant serving `'North Indian, Chinese, Biryani'` scores 3.

In [6]:
def count_cuisines(val):
    if pd.isna(val) or str(val).strip().lower() == 'unknown':
        return 0
    return len([c for c in str(val).split(',') if c.strip()])

df['cuisine_count'] = df['cuisines'].apply(count_cuisines)

print('cuisine_count statistics:')
print(df['cuisine_count'].describe())
print(f'\nZero cuisine_count (Unknown or null): {(df["cuisine_count"] == 0).sum():,}')
print(f'Max cuisines in one restaurant     : {df["cuisine_count"].max()}')
print('\nTop 10 cuisine counts:')
print(df['cuisine_count'].value_counts().head(10))

cuisine_count statistics:
count   51717.0000
mean        2.4522
std         1.2718
min         0.0000
25%         2.0000
50%         2.0000
75%         3.0000
max         8.0000
Name: cuisine_count, dtype: float64

Zero cuisine_count (Unknown or null): 45
Max cuisines in one restaurant     : 8

Top 10 cuisine counts:
cuisine_count
2    17920
1    12402
3    12172
4     5869
5     2044
6      680
7      395
8      190
0       45
Name: count, dtype: int64


### 4.3 · Dish Count
Number of popular dishes listed in `dish_liked`. Restaurants with more popular dishes have a richer menu signal.

In [7]:
def count_dishes(val):
    if pd.isna(val) or str(val).strip() == '':
        return 0
    return len([d for d in str(val).split(',') if d.strip()])

df['dish_count'] = df['dish_liked'].apply(count_dishes)

print('dish_count statistics:')
print(df['dish_count'].describe())
print(f'\nRestaurants with no dish data: {(df["dish_count"] == 0).sum():,}')
print(f'Max dishes listed            : {df["dish_count"].max()}')

dish_count statistics:
count   51717.0000
mean        2.4914
std         3.0730
min         0.0000
25%         0.0000
50%         0.0000
75%         7.0000
max         7.0000
Name: dish_count, dtype: float64

Restaurants with no dish data: 28,078
Max dishes listed            : 7


### 4.4 · Review Count
Number of individual reviews inside `reviews_list`. Zomato stores reviews with a `Rated X.X` prefix per entry — counting those occurrences gives the review volume per restaurant.

In [8]:
def count_reviews(val):
    """
    Count the number of individual reviews by counting occurrences of the
    'Rated' prefix that Zomato prepends to every review entry.
    Falls back to 0 for null or empty values.
    """
    if pd.isna(val) or str(val).strip() == '':
        return 0
    return len(re.findall(r'Rated', str(val)))

df['review_count'] = df['reviews_list'].apply(count_reviews)

print('review_count statistics:')
print(df['review_count'].describe())
print(f'\nRestaurants with 0 reviews : {(df["review_count"] == 0).sum():,}')
print(f'Max reviews in one entry   : {df["review_count"].max()}')

# Sanity check — review_count should broadly correlate with votes
corr = df[['votes', 'review_count']].corr().iloc[0, 1]
print(f'\nCorrelation (votes vs review_count): {corr:.4f}  (expected: positive)')

review_count statistics:
count   51717.0000
mean       25.5252
std        71.9737
min         0.0000
25%         1.0000
50%         3.0000
75%        11.0000
max      1769.0000
Name: review_count, dtype: float64

Restaurants with 0 reviews : 7,595
Max reviews in one entry   : 1769

Correlation (votes vs review_count): 0.3641  (expected: positive)


### 4.5 · Restaurant Popularity Index (RPI)

**Why RPI instead of raw votes or rating alone:**  
A restaurant with a 4.8 rating from 15 customers is not necessarily better than one with a 4.5 rating from 3,500 customers. The RPI captures both **customer satisfaction** (`rate`) and **customer confidence** (`votes`) in a single feature. The logarithmic weighting prevents restaurants with extreme vote counts from dominating the score.

```
RPI = rate × log(1 + votes)
```

| Restaurant | Rate | Votes | RPI |
|-----------|------|-------|-----|
| A | 4.8 | 15 | 4.8 × log(16) ≈ 13.4 |
| B | 4.5 | 3500 | 4.5 × log(3501) ≈ 37.2 |

Restaurant B has stronger evidence from thousands of customers — RPI reflects that balance.

> **Why RPI is `NaN` for unrated restaurants:** Popularity is a composite of customer confidence (`votes`) and customer satisfaction (`rate`). Without a rating, only half the signal exists — a high vote count alone without any quality score cannot produce a meaningful composite. RPI is therefore **intentionally undefined** for unrated restaurants. This is not a data quality issue; it is a deliberate design decision. These rows are retained in the full dataset for NLP, recommendation, and RAG use cases.

In [9]:
# RPI requires valid rate — rows with null rate get null RPI (correct behaviour)
df['rpi'] = df['rate'] * np.log1p(df['votes'])

print('RPI (Restaurant Popularity Index) statistics:')
print(df['rpi'].describe())
print(f'\nNull RPI (intentionally undefined — missing rate): {df["rpi"].isnull().sum():,}')

# Show top 10 restaurants by RPI
print('\nTop 10 restaurants by RPI:')
top_rpi = (
    df[['name', 'rate', 'votes', 'rpi']]
    .dropna(subset=['rpi'])
    .sort_values('rpi', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print(top_rpi.to_string())

RPI (Restaurant Popularity Index) statistics:
count   41665.0000
mean       16.8608
std         7.8862
min         0.0000
25%        10.6000
50%        15.5889
75%        22.1018
max        47.6824
Name: rpi, dtype: float64

Null RPI (intentionally undefined — missing rate): 10,052

Top 10 restaurants by RPI:
                          name   rate  votes     rpi
0  Byg Brewski Brewing Company 4.9000  16832 47.6824
1  Byg Brewski Brewing Company 4.9000  16832 47.6824
2  Byg Brewski Brewing Company 4.9000  16832 47.6824
3  Byg Brewski Brewing Company 4.9000  16345 47.5385
4  Byg Brewski Brewing Company 4.9000  16345 47.5385
5  Byg Brewski Brewing Company 4.9000  16345 47.5385
6                         Toit 4.7000  14956 45.1808
7                         Toit 4.7000  14956 45.1808
8    AB's - Absolute Barbecues 4.8000  12121 45.1333
9    AB's - Absolute Barbecues 4.8000  12121 45.1333


### 4.6 · Engineered Feature Summary

In [10]:
ENGINEERED_COLS = ['rating_category', 'cost_category', 'cuisine_count',
                   'dish_count', 'review_count', 'rpi']

print('All engineered features:')
print(f'{"Feature":<20} {"Dtype":<12} {"Nulls":>8} {"Valid":>10} {"Unique":>8}')
print('-' * 62)
for col in ENGINEERED_COLS:
    nulls = df[col].isnull().sum()
    valid = df[col].notna().sum()
    print(f'{col:<20} {str(df[col].dtype):<12} {nulls:>8,} {valid:>10,} {df[col].nunique():>8}')

print()
print('NOTE: Only `rpi` is expected to contain missing values.')
print('      All other engineered features should be fully populated.')
print(f'      rpi is NaN for {df["rpi"].isnull().sum():,} restaurants that lack a rating (intentional).')
print(f'\nDataset shape after feature creation: {df.shape}')

All engineered features:
Feature              Dtype           Nulls      Valid   Unique
--------------------------------------------------------------
rating_category      category       10,052     41,665        4
cost_category        category            0     51,717        4
cuisine_count        int64               0     51,717        9
dish_count           int64               0     51,717        8
review_count         int64               0     51,717      626
rpi                  float64        10,052     41,665     7268

NOTE: Only `rpi` is expected to contain missing values.
      All other engineered features should be fully populated.
      rpi is NaN for 10,052 restaurants that lack a rating (intentional).

Dataset shape after feature creation: (51717, 20)


## 5 · Categorical Feature Encoding

**Why this step exists:**  
Machine learning algorithms operate on numbers, not strings. However, the encoding strategy must match the downstream algorithm — there is no single correct encoding for all models.

- **Tree-based algorithms** (Decision Tree, LightGBM) handle integer label-encoded categories efficiently without creating sparse matrices. They split on thresholds and do not interpret integer labels as having ordinal meaning, so label encoding is mathematically safe.
- **Support Vector Machines** rely on distance calculations in a high-dimensional kernel space. Arbitrary integer labels (e.g., location 0 = "Indiranagar", location 1 = "Koramangala") imply a false ordering that distorts distances. One-Hot Encoding gives each category its own binary dimension, eliminating this bias.

Applying the wrong encoding to the wrong model type is a common interview failure point. Both sets are constructed here so that no model notebook repeats this logic.

| Encoding | Applied To | Models |
|----------|-----------|--------|
| Binary (0/1) | `online_order`, `book_table` | All |
| Label Encoding | `location`, `rest_type`, `listed_in(type)`, `listed_in(city)` | Tree + LightGBM |
| One-Hot Encoding | `location`, `rest_type`, `listed_in(city)`, `listed_in(type)` | SVM |
| No encoding yet | `cuisines`, `reviews_list`, `dish_liked`, `menu_item` | NLP / RAG |

### 5.1 · Binary Encoding — `online_order`, `book_table`

In [11]:
BINARY_MAP = {'Yes': 1, 'No': 0}

for col in BINARY_COLS:
    encoded_col = col.lower().replace(' ', '_').replace('(', '').replace(')', '') + '_enc'
    df[encoded_col] = df[col].map(BINARY_MAP)
    print(f'{col} → {encoded_col}')
    print(f'  Value counts: {df[encoded_col].value_counts().to_dict()}')
    print(f'  Nulls       : {df[encoded_col].isnull().sum()}')

# Convenience references
ONLINE_ORDER_ENC = 'online_order_enc'
BOOK_TABLE_ENC   = 'book_table_enc'

online_order → online_order_enc
  Value counts: {1: 30444, 0: 21273}
  Nulls       : 0
book_table → book_table_enc
  Value counts: {0: 45268, 1: 6449}
  Nulls       : 0


### 5.2 · Label Encoding — for Tree-Based Models
Label encoding assigns an integer to each unique category. Appropriate for Decision Tree and LightGBM because these algorithms split on thresholds and do not interpret integers as having ordinal meaning.

In [12]:
label_encoders   = {}   # store encoders so they can be reused in model notebooks
LABEL_ENC_RESULT = {}   # map original col → encoded col name

for col in LABEL_ENC_COLS:
    enc_col = col.replace('(', '').replace(')', '').replace(' ', '_').lower() + '_enc'
    le = LabelEncoder()
    # fillna with 'Unknown' before encoding so NaN does not cause errors
    df[enc_col] = le.fit_transform(df[col].fillna('Unknown').astype(str))
    label_encoders[col]   = le
    LABEL_ENC_RESULT[col] = enc_col
    print(f'{col:<35} → {enc_col:<40} | {df[col].nunique()} classes')

print('\nLabel encoders stored for reuse in model notebooks.')

listed_in(type)                     → listed_intype_enc                        | 7 classes
listed_in(city)                     → listed_incity_enc                        | 30 classes
location                            → location_enc                             | 94 classes
rest_type                           → rest_type_enc                            | 94 classes

Label encoders stored for reuse in model notebooks.


### 5.3 · Classification Target — Label Encode `rating_category`

In [13]:
# Encode rating_category for classification models
# Order: Poor=0, Average=1, Good=2, Excellent=3
RATING_CAT_MAP = {'Poor': 0, 'Average': 1, 'Good': 2, 'Excellent': 3}

df['rating_category_enc'] = df['rating_category'].map(RATING_CAT_MAP)

print('rating_category_enc distribution:')
print(df['rating_category_enc'].value_counts(dropna=False).sort_index())
print(f'\nNull (from missing rate): {df["rating_category_enc"].isnull().sum():,}')
print(f'\nMapping: {RATING_CAT_MAP}')

rating_category_enc distribution:
rating_category_enc
0        288
1      13996
2      23297
3       4084
NaN    10052
Name: count, dtype: int64

Null (from missing rate): 10,052

Mapping: {'Poor': 0, 'Average': 1, 'Good': 2, 'Excellent': 3}


### 5.4 · Cost Category — Label Encode

In [14]:
COST_CAT_MAP = {'Budget': 0, 'Mid-Range': 1, 'Premium': 2, 'Luxury': 3}

df['cost_category_enc'] = df['cost_category'].map(COST_CAT_MAP)

print('cost_category_enc distribution:')
print(df['cost_category_enc'].value_counts(dropna=False).sort_index())
print(f'\nNull (from missing cost_category): {df["cost_category_enc"].isnull().sum():,}')
print(f'\nMapping: {COST_CAT_MAP}')

cost_category_enc distribution:
cost_category_enc
0    18554
1    19551
2     5767
3     7845
Name: count, dtype: int64

Null (from missing cost_category): 0

Mapping: {'Budget': 0, 'Mid-Range': 1, 'Premium': 2, 'Luxury': 3}


## 6 · Numerical Feature Preparation

**Why this step exists:**  
Two numerical issues are addressed here. First, `approx_cost(for two people)` is **validated** — median imputation was already applied during Phase 2 cleaning, so this section confirms the column is fully populated before any modelling. Second, `votes` has extreme outliers (max=16,832, mean≈260) that will dominate distance-based and gradient calculations. A log transformation compresses the tail without removing the information carried by high-vote restaurants.

> **Why no imputation here:** Repeating imputation that was already performed in `02_Data_Cleaning.ipynb` would make the pipeline harder to understand and harder to maintain. Phase 2 is responsible for data completeness; Phase 3 is responsible for feature creation only.

### 6.1 · Validation — `approx_cost(for two people)`

> Median imputation for `approx_cost(for two people)` was completed in `02_Data_Cleaning.ipynb`. This section verifies that the column contains no missing values and reports the post-cleaning statistics.

In [15]:
# ── Validate approx_cost — imputation was done in Phase 2 ────────────────
null_count = df['approx_cost(for two people)'].isnull().sum()

print(f'approx_cost null count : {null_count}')
print(f'Status                 : {"✅ CLEAN — no imputation needed here" if null_count == 0 else "❌ UNEXPECTED NULLS — check 02_Data_Cleaning.ipynb"}')
print()
print('approx_cost(for two people) statistics (post-cleaning):')
print(df['approx_cost(for two people)'].describe())

# Store median for reference in the impact report (read from data, not recomputed)
cost_median = df['approx_cost(for two people)'].median()
print(f'\nMedian (for reference): {cost_median:.2f}')

approx_cost null count : 0
Status                 : ✅ CLEAN — no imputation needed here

approx_cost(for two people) statistics (post-cleaning):
count   51717.0000
mean      554.3917
std       437.5637
min        40.0000
25%       300.0000
50%       400.0000
75%       650.0000
max      6000.0000
Name: approx_cost(for two people), dtype: float64

Median (for reference): 400.00


### 6.2 · Log Transform — `votes`
Votes range from 0 to 16,832 with 13.45% outliers by IQR. `log1p` (log(1 + x)) compresses the scale without losing zero-vote information.

In [16]:
df['votes_log'] = np.log1p(df['votes'])

print('votes vs votes_log comparison:')
print(f'{"Stat":<12} {"votes":>12} {"votes_log":>12}')
print('-' * 38)
for stat in ['min', 'mean', 'median', 'std', 'max']:
    rv = getattr(df['votes'],     stat)()
    lv = getattr(df['votes_log'], stat)()
    print(f'{stat:<12} {rv:>12.2f} {lv:>12.4f}')

print(f'\nvotes_log nulls: {df["votes_log"].isnull().sum()}')

votes vs votes_log comparison:
Stat                votes    votes_log
--------------------------------------
min                  0.00       0.0000
mean               283.70       3.5714
median              41.00       3.7377
std                803.84       2.3159
max              16832.00       9.7311

votes_log nulls: 0


## 7 · Feature Sets — Tree vs SVM

**Why this step exists:**  
Decision Tree and LightGBM are scale-invariant — they split on feature thresholds and integer-encoded categories work correctly as-is. SVM uses distance between data points in a high-dimensional space, so features on different scales will cause some dimensions to dominate others. This step constructs two clean feature matrices: one for tree-based models and one for SVM, so that neither set of model notebooks needs to repeat this logic.

In [17]:
# ── Feature columns for tree-based models ────────────────────────────────
# Uses label-encoded categoricals + log-transformed votes + all engineered features

TREE_FEATURES = [
    # Encoded binary
    'online_order_enc', 'book_table_enc',
    # Encoded categorical
    'location_enc', 'rest_type_enc',
    'listed_intype_enc', 'listed_incity_enc',
    # Numerical (original + log)
    'approx_cost(for two people)', 'votes', 'votes_log',
    # Engineered
    'cuisine_count', 'dish_count', 'review_count', 'rpi',
    'cost_category_enc',
]

# Verify all columns exist
missing_tree = [c for c in TREE_FEATURES if c not in df.columns]
if missing_tree:
    print(f'⚠️  Missing tree feature columns: {missing_tree}')
else:
    print(f'Tree feature set ready — {len(TREE_FEATURES)} features')
    print(TREE_FEATURES)

Tree feature set ready — 14 features
['online_order_enc', 'book_table_enc', 'location_enc', 'rest_type_enc', 'listed_intype_enc', 'listed_incity_enc', 'approx_cost(for two people)', 'votes', 'votes_log', 'cuisine_count', 'dish_count', 'review_count', 'rpi', 'cost_category_enc']


In [18]:
# ── One-Hot Encoding for SVM ──────────────────────────────────────────────
# SVM needs proper numeric spread for categorical columns
# location (93+1 unique), rest_type (93+1 unique), listed_in(city) (30+1)
# listed_in(type) (7+1) — label encoding is acceptable for low cardinality

print('Applying One-Hot Encoding for SVM pipeline...')
print(f'  OHE columns   : {OHE_COLS}')
print(f'  Unique counts : { {col: df[col].nunique() for col in OHE_COLS} }')

df_svm_ohe = pd.get_dummies(
    df[OHE_COLS],
    prefix     = OHE_COLS,
    drop_first = True,    # avoid perfect multicollinearity
    dtype      = int
)

print(f'\nOHE shape: {df_svm_ohe.shape}')
print(f'Columns generated by OHE: {len(df_svm_ohe.columns)}')

# Numeric + binary + engineered features for SVM (no label-encoded high-cardinality cols)
SVM_NUMERIC_FEATURES = [
    'online_order_enc', 'book_table_enc',
    'listed_intype_enc',                     # low cardinality — label OK for SVM
    'approx_cost(for two people)', 'votes_log',
    'cuisine_count', 'dish_count', 'review_count', 'rpi',
    'cost_category_enc',
]

df_svm_base = pd.concat(
    [df[SVM_NUMERIC_FEATURES].reset_index(drop=True),
     df_svm_ohe.reset_index(drop=True)],
    axis=1
)

print(f'\nSVM base feature matrix shape (before scaling): {df_svm_base.shape}')

Applying One-Hot Encoding for SVM pipeline...
  OHE columns   : ['location', 'rest_type', 'listed_in(city)']
  Unique counts : {'location': 94, 'rest_type': 94, 'listed_in(city)': 30}

OHE shape: (51717, 215)
Columns generated by OHE: 215

SVM base feature matrix shape (before scaling): (51717, 225)


## 8 · Feature Scaling — SVM Only

**Why this step exists:**  
StandardScaler transforms each numerical feature to zero mean and unit variance. This is **mathematically required** for SVM because the kernel function computes distances between samples — unscaled features with large ranges (e.g., votes 0–16,832) would dominate features with small ranges (e.g., cuisine_count 0–35), causing the SVM to learn a distorted decision boundary.

Decision Tree and LightGBM are **not scaled** because:
- They split on individual feature thresholds, not distances
- Scaling would not improve their performance
- Keeping unscaled data makes tree model outputs more interpretable

> **Preventing data leakage:** The StandardScaler is fitted **exclusively on the training set** and then reused (`.transform()` only) for the test set. This ensures that no distributional information from unseen test data influences the learned feature statistics. Fitting the scaler on the full dataset before splitting is a subtle but serious form of leakage.

In [19]:
# Scaling is performed AFTER the train-test split in section 10
# This cell just validates the SVM feature matrix is numerically clean

print('SVM feature matrix pre-scaling validation:')
print(f'  Shape   : {df_svm_base.shape}')
print(f'  Nulls   : {df_svm_base.isnull().sum().sum()}')
print(f'  Dtypes  : {df_svm_base.dtypes.value_counts().to_dict()}')
print('\nColumn ranges (sample):')
print(df_svm_base[SVM_NUMERIC_FEATURES].describe().loc[['min','mean','max']].T)

SVM feature matrix pre-scaling validation:
  Shape   : (51717, 225)
  Nulls   : 10052
  Dtypes  : {dtype('int64'): 221, dtype('float64'): 3, CategoricalDtype(categories=[0, 1, 2, 3], ordered=True, categories_dtype=int64): 1}

Column ranges (sample):
                                min     mean       max
online_order_enc             0.0000   0.5887    1.0000
book_table_enc               0.0000   0.1247    1.0000
listed_intype_enc            0.0000   2.8074    6.0000
approx_cost(for two people) 40.0000 554.3917 6000.0000
votes_log                    0.0000   3.5714    9.7311
cuisine_count                0.0000   2.4522    8.0000
dish_count                   0.0000   2.4914    7.0000
review_count                 0.0000  25.5252 1769.0000
rpi                          0.0000  16.8608   47.6824


## 9 · NLP Dataset Preservation

**Why this step exists:**  
The text columns (`reviews_list`, `dish_liked`, `menu_item`) will be transformed into numerical representations using TF-IDF or embedding-based methods in dedicated NLP notebooks. They must not be encoded or modified here. They are extracted now as a clean, aligned dataset with the restaurant identifier, rating, and key metadata columns.

> `location`, `cuisines`, and `rest_type` are included because they are critical for contextual RAG queries such as *"Recommend North Indian restaurants in Indiranagar under ₹500."* Including them here avoids repeated cross-notebook joins in later pipelines.

In [20]:
NLP_COLS = [
    'name', 'rate', 'rating_category', 'votes', 'rpi',
    # Context columns for RAG / recommendation
    'location', 'cuisines', 'rest_type',
    # Text columns for NLP
    'reviews_list', 'dish_liked', 'menu_item',
]

df_nlp = df[NLP_COLS].copy()

print('NLP dataset:')
print(f'  Shape   : {df_nlp.shape}')
print(f'  Columns : {df_nlp.columns.tolist()}')
print('\nNull counts in NLP dataset:')
print(df_nlp.isnull().sum())

NLP dataset:
  Shape   : (51717, 11)
  Columns : ['name', 'rate', 'rating_category', 'votes', 'rpi', 'location', 'cuisines', 'rest_type', 'reviews_list', 'dish_liked', 'menu_item']

Null counts in NLP dataset:
name                   0
rate               10052
rating_category    10052
votes                  0
rpi                10052
location               0
cuisines               0
rest_type              0
reviews_list           0
dish_liked         28078
menu_item              0
dtype: int64


## 10 · Train-Test Split

**Why this step exists:**  
Splitting the data here — before any model sees it — ensures that all three model types (Decision Tree, LightGBM, SVM) are evaluated on identical held-out test rows. Using `random_state=42` everywhere makes the split deterministic. The same split is reused across all model notebooks so that performance comparisons are fair.

> **This is the first and only point in the entire pipeline where restaurants without a valid `rate` are excluded.** Every notebook before this one — including `02_Data_Cleaning.ipynb` — intentionally preserved those rows because they carry value for non-supervised tasks. Supervised learning requires a known target label, which is why the exclusion happens here and not earlier. Restaurants with a missing `rate` remain in `df` (the full engineered dataset) and in `df_nlp`, so they continue to flow into recommendation, RAG, and Flask pipelines.

- 80% training / 20% testing
- StandardScaler fitted on training data only and applied to test data — no leakage
- Stratified split used for classification to preserve class proportions

In [21]:
# ── Rows with valid rate only (supervised learning) ───────────────────────
df_supervised = df[df['rate'].notna()].copy().reset_index(drop=True)
df_svm_base_sup = df_svm_base[df['rate'].notna()].copy().reset_index(drop=True)

total_rows      = len(df)
supervised_rows = len(df_supervised)
unrated_rows    = total_rows - supervised_rows

print(f'Total rows (full engineered dataset)  : {total_rows:,}')
print(f'Rows with valid rate (supervised)     : {supervised_rows:,}')
print(f'Rows without rate (Flask/NLP/RAG use) : {unrated_rows:,}')
print()
print('Unrated restaurants are NOT lost — they flow into:')
print('  ✅ zomato_engineered_full.csv  (recommendation + Flask)')
print('  ✅ zomato_nlp_dataset.csv      (NLP + RAG pipelines)')

Total rows (full engineered dataset)  : 51,717
Rows with valid rate (supervised)     : 41,665
Rows without rate (Flask/NLP/RAG use) : 10,052

Unrated restaurants are NOT lost — they flow into:
  ✅ zomato_engineered_full.csv  (recommendation + Flask)
  ✅ zomato_nlp_dataset.csv      (NLP + RAG pipelines)


In [22]:
# ── Regression split — tree models ────────────────────────────────────────
X_tree = df_supervised[TREE_FEATURES]
y_reg  = df_supervised['rate']

X_train_tree, X_test_tree, y_train_reg, y_test_reg = train_test_split(
    X_tree, y_reg,
    test_size    = 0.20,
    random_state = RANDOM_STATE
)

print('Regression split (tree):')
print(f'  X_train : {X_train_tree.shape}  |  y_train : {y_train_reg.shape}')
print(f'  X_test  : {X_test_tree.shape}   |  y_test  : {y_test_reg.shape}')
print(f'  y_train mean: {y_train_reg.mean():.4f}  |  y_test mean: {y_test_reg.mean():.4f}')

Regression split (tree):
  X_train : (33332, 14)  |  y_train : (33332,)
  X_test  : (8333, 14)   |  y_test  : (8333,)
  y_train mean: 3.7018  |  y_test mean: 3.6952


In [23]:
# ── Classification split — tree models ────────────────────────────────────
y_clf = df_supervised['rating_category_enc']

X_train_tree_clf, X_test_tree_clf, y_train_clf, y_test_clf = train_test_split(
    X_tree, y_clf,
    test_size    = 0.20,
    random_state = RANDOM_STATE,
    stratify     = y_clf   # preserve class proportions
)

print('Classification split (tree):')
print(f'  X_train : {X_train_tree_clf.shape}  |  y_train : {y_train_clf.shape}')
print(f'  X_test  : {X_test_tree_clf.shape}   |  y_test  : {y_test_clf.shape}')
print('\ny_train class distribution:')
print(y_train_clf.value_counts().sort_index())
print('\ny_test class distribution:')
print(y_test_clf.value_counts().sort_index())

Classification split (tree):
  X_train : (33332, 14)  |  y_train : (33332,)
  X_test  : (8333, 14)   |  y_test  : (8333,)

y_train class distribution:
rating_category_enc
0      230
1    11197
2    18638
3     3267
Name: count, dtype: int64

y_test class distribution:
rating_category_enc
0      58
1    2799
2    4659
3     817
Name: count, dtype: int64


In [24]:
# ── SVM split + fit-transform scaling ─────────────────────────────────────
X_svm = df_svm_base_sup
y_svm = df_supervised['rate']

X_train_svm_raw, X_test_svm_raw, y_train_svm, y_test_svm = train_test_split(
    X_svm, y_svm,
    test_size    = 0.20,
    random_state = RANDOM_STATE
)

# ── Null audit before scaling ─────────────────────────────────────────────
pre_scale_nulls = X_train_svm_raw.isnull().sum()
pre_scale_nulls = pre_scale_nulls[pre_scale_nulls > 0]

if len(pre_scale_nulls) > 0:
    print('Nulls found in X_train_svm_raw BEFORE scaling:')
    print(pre_scale_nulls)
    print('\nFilling residual nulls with column median...')
    col_medians = X_train_svm_raw.median()
    X_train_svm_raw = X_train_svm_raw.fillna(col_medians)
    X_test_svm_raw  = X_test_svm_raw.fillna(col_medians)   # use train medians for test
    print('Done.')
else:
    print('No nulls in X_train_svm_raw — clean going into scaler.')

# ── Scale — fit on train only, transform test with same learned stats ──────
scaler = StandardScaler()
X_train_svm = pd.DataFrame(
    scaler.fit_transform(X_train_svm_raw),   # fit on train only
    columns = X_train_svm_raw.columns
)
X_test_svm  = pd.DataFrame(
    scaler.transform(X_test_svm_raw),        # transform test using train stats only
    columns = X_test_svm_raw.columns
)

print(f'\nX_train_svm : {X_train_svm.shape}  |  nulls: {X_train_svm.isnull().sum().sum()}')
print(f'X_test_svm  : {X_test_svm.shape}   |  nulls: {X_test_svm.isnull().sum().sum()}')
print(f'\nScaler fitted on training data only — test set uses train mean/std.')
print(f'Train mean of first 3 features (should be ~0 after scaling):')
print(X_train_svm.iloc[:, :3].mean().round(4).to_dict())

No nulls in X_train_svm_raw — clean going into scaler.

X_train_svm : (33332, 225)  |  nulls: 0
X_test_svm  : (8333, 225)   |  nulls: 0

Scaler fitted on training data only — test set uses train mean/std.
Train mean of first 3 features (should be ~0 after scaling):
{'online_order_enc': 0.0, 'book_table_enc': 0.0, 'listed_intype_enc': -0.0}


## 11 · Assertions

**Why this step exists:**  
Feature engineering introduces more opportunities for silent bugs than cleaning does — a wrong column reference, a merge that drops rows, or an encoder that silently maps unknowns to -1. These assertions act as a formal test suite: if any check fails, the notebook stops before exporting a corrupted dataset.

In [25]:
print('Running feature engineering assertions...')

# ── approx_cost — must be fully clean (imputed in Phase 2) ───────────────
assert df['approx_cost(for two people)'].isnull().sum() == 0, \
    'approx_cost still has nulls — check 02_Data_Cleaning.ipynb'
assert df['approx_cost(for two people)'].min() >= 0, \
    'approx_cost has negative values'

# ── Imputed string columns — must have no nulls ───────────────────────────
assert df['location'].isnull().sum() == 0, \
    'location has nulls — should be imputed with "Unknown" in Phase 2'
assert df['rest_type'].isnull().sum() == 0, \
    'rest_type has nulls — should be imputed with "Unknown" in Phase 2'
assert df['cuisines'].isnull().sum() == 0, \
    'cuisines has nulls — should be imputed with "Unknown" in Phase 2'

# ── Target variable ────────────────────────────────────────────────────────
assert df['rate'].dtype == np.float64, \
    f'rate should be float64, got {df["rate"].dtype}'
assert df['rate'].dropna().between(0, 5).all(), \
    f'rate has values outside [0, 5]'
assert set(df['rating_category'].dropna().unique()) == set(RATING_LABELS), \
    f'Unexpected rating_category values: {df["rating_category"].unique()}'
assert df['rating_category_enc'].dropna().between(0, 3).all(), \
    'rating_category_enc has values outside [0, 3]'

# ── Binary encoding ────────────────────────────────────────────────────────
assert set(df['online_order_enc'].dropna().unique()).issubset({0, 1}), \
    'online_order_enc has values other than 0 and 1'
assert set(df['book_table_enc'].dropna().unique()).issubset({0, 1}), \
    'book_table_enc has values other than 0 and 1'
assert df['online_order_enc'].isnull().sum() == 0, \
    'online_order_enc has unexpected nulls'

# ── Engineered features ────────────────────────────────────────────────────
assert (df['cuisine_count'] >= 0).all(), 'cuisine_count has negative values'
assert (df['dish_count'] >= 0).all(),    'dish_count has negative values'
assert (df['review_count'] >= 0).all(),  'review_count has negative values'
assert (df['votes_log'] >= 0).all(),     'votes_log has negative values'
assert df['rpi'].dropna().min() >= 0,    'RPI has negative values'

# ── cost_category — should be fully populated (no nulls since approx_cost is clean) ──
assert df['cost_category'].isnull().sum() == 0, \
    'cost_category has nulls — approx_cost should have been imputed in Phase 2'

# ── Train-test split integrity ─────────────────────────────────────────────
assert len(X_train_tree) + len(X_test_tree) == len(df_supervised), \
    'Train + test rows do not sum to supervised dataset size'
assert abs(len(X_test_tree) / len(df_supervised) - 0.20) < 0.01, \
    f'Test set is not approximately 20%: {len(X_test_tree)/len(df_supervised):.2%}'

# ── SVM scaling check ──────────────────────────────────────────────────────
assert X_train_svm.isnull().sum().sum() == 0, 'X_train_svm has null values after scaling'
assert X_test_svm.isnull().sum().sum() == 0,  'X_test_svm has null values after scaling'

# ── Required engineered columns exist ─────────────────────────────────────
REQUIRED_NEW_COLS = [
    'rating_category', 'rating_category_enc',
    'cost_category', 'cost_category_enc',
    'cuisine_count', 'dish_count', 'review_count',
    'rpi', 'votes_log',
    'online_order_enc', 'book_table_enc',
]
for col in REQUIRED_NEW_COLS:
    assert col in df.columns, f'Required engineered column missing: {col}'

print('✅ All assertions passed. Feature engineering output is valid.')

Running feature engineering assertions...
✅ All assertions passed. Feature engineering output is valid.


## 12 · Feature Engineering Impact Report

A structured audit of every transformation applied in this notebook.

In [26]:
sep  = '=' * 70
sep2 = '-' * 70

print(sep)
print('  ZOMATO DATASET — FEATURE ENGINEERING IMPACT REPORT')
print(sep)

# ── Row summary (CHANGE 12) ───────────────────────────────────────────────
print(f'\n  {"ROW SUMMARY":}')
print(f'  {sep2}')
print(f'  Original cleaned rows            : {df_clean.shape[0]:,}')
print(f'  Rows after feature engineering   : {len(df):,}  (no rows dropped)')
print(f'  Rows for supervised ML           : {len(df_supervised):,}  (valid rate)')
print(f'  Rows retained for Flask/RAG/NLP  : {len(df) - len(df_supervised):,}  (missing rate — intentionally preserved)')

print(f'\n  Input  : {INPUT_PATH.name}  |  {df_clean.shape[0]:,} rows × {df_clean.shape[1]} cols')
print(f'  Features in tree model set     : {len(TREE_FEATURES)}')
print(f'  Features in SVM model set      : {df_svm_base.shape[1]}')

print(f'\n  {sep2}')
print('  ENGINEERED FEATURES')
print(f'  {sep2}')
fe_log = [
    ('rating_category',     'rate',                        'pd.cut → Poor/Average/Good/Excellent'),
    ('rating_category_enc', 'rating_category',             'Map → 0/1/2/3 (classification target)'),
    ('cost_category',       'approx_cost(for two people)', 'pd.cut → Budget/Mid-Range/Premium/Luxury'),
    ('cost_category_enc',   'cost_category',               'Map → 0/1/2/3'),
    ('cuisine_count',       'cuisines',                    'Count comma-separated items'),
    ('dish_count',          'dish_liked',                  'Count comma-separated items'),
    ('review_count',        'reviews_list',                'Count "Rated" prefix occurrences'),
    ('votes_log',           'votes',                       'np.log1p — compresses skewed tail'),
    ('rpi',                 'rate + votes',                'rate × log(1 + votes) — Popularity Index'),
]
print(f'  {"New Feature":<22} {"Source":<35} {"Method"}')
print(f'  {sep2}')
for feat, source, method in fe_log:
    print(f'  {feat:<22} {source:<35} {method}')

print(f'\n  {sep2}')
print('  ENCODING LOG')
print(f'  {sep2}')
enc_log = [
    ('online_order',    'Binary Map',     'Yes→1, No→0',                                  'All models'),
    ('book_table',      'Binary Map',     'Yes→1, No→0',                                  'All models'),
    ('location',        'LabelEncoder',   '93+ classes',                                  'Tree / LightGBM'),
    ('rest_type',       'LabelEncoder',   '93+ classes',                                  'Tree / LightGBM'),
    ('listed_in(type)', 'LabelEncoder',   '7 classes',                                    'Tree + SVM'),
    ('listed_in(city)', 'LabelEncoder',   '30 classes',                                   'Tree / LightGBM'),
    ('location',        'get_dummies OHE', f'{df_svm_ohe.filter(like="location").shape[1]} cols', 'SVM'),
    ('rest_type',       'get_dummies OHE', f'{df_svm_ohe.filter(like="rest_type").shape[1]} cols', 'SVM'),
    ('listed_in(city)', 'get_dummies OHE', f'{df_svm_ohe.filter(like="listed_in(city)").shape[1]} cols', 'SVM'),
]
print(f'  {"Column":<18} {"Method":<17} {"Result":<20} {"Used By"}')
print(f'  {sep2}')
for col, method, result, used_by in enc_log:
    print(f'  {col:<18} {method:<17} {result:<20} {used_by}')

print(f'\n  {sep2}')
print('  NUMERICAL PREPARATION')
print(f'  {sep2}')
print(f'  approx_cost   : Imputed in Phase 2 — validated clean here (0 nulls)')
print(f'  votes         : log1p transformation → votes_log')
print(f'  StandardScaler: fitted on X_train_svm only — no test leakage')

print(f'\n  {sep2}')
print('  TRAIN-TEST SPLIT')
print(f'  {sep2}')
print(f'  Strategy      : 80/20, random_state={RANDOM_STATE}, stratified for classification')
print(f'  X_train_tree  : {X_train_tree.shape}')
print(f'  X_test_tree   : {X_test_tree.shape}')
print(f'  X_train_svm   : {X_train_svm.shape}  (scaled)')
print(f'  X_test_svm    : {X_test_svm.shape}   (scaled)')

print(f'\n  {sep2}')
print('  DEFERRED TO NLP NOTEBOOKS')
print(f'  {sep2}')
print('  reviews_list  : TF-IDF / embeddings — NLP notebook')
print('  dish_liked    : TF-IDF — dish-level NLP analysis')
print('  menu_item     : RAG pipeline — chunked retrieval')

print(f'\n{sep}')

  ZOMATO DATASET — FEATURE ENGINEERING IMPACT REPORT

  ROW SUMMARY
  ----------------------------------------------------------------------
  Original cleaned rows            : 51,717
  Rows after feature engineering   : 51,717  (no rows dropped)
  Rows for supervised ML           : 41,665  (valid rate)
  Rows retained for Flask/RAG/NLP  : 10,052  (missing rate — intentionally preserved)

  Input  : zomato_cleaned_v1.csv  |  51,717 rows × 14 cols
  Features in tree model set     : 14
  Features in SVM model set      : 225

  ----------------------------------------------------------------------
  ENGINEERED FEATURES
  ----------------------------------------------------------------------
  New Feature            Source                              Method
  ----------------------------------------------------------------------
  rating_category        rate                                pd.cut → Poor/Average/Good/Excellent
  rating_category_enc    rating_category                     Ma

## 13 · Data Dictionary — Engineered Dataset

In [27]:
data_dict = [
    # Original columns (retained)
    ('name',                         'object',  'Identifier',  'Reference',       'Restaurant name — not used as feature'),
    ('rate',                         'float64', 'Target',      'Regression',      'Continuous target — null rows excluded at train time'),
    ('votes',                        'int64',   'Numerical',   'Tree + LightGBM', 'Raw vote count — use votes_log for models'),
    ('online_order',                 'object',  'Binary',      'Reference',       'Original — use online_order_enc for models'),
    ('book_table',                   'object',  'Binary',      'Reference',       'Original — use book_table_enc for models'),
    ('location',                     'object',  'Categorical', 'Reference',       'Original — use location_enc or OHE for models'),
    ('rest_type',                    'object',  'Categorical', 'Reference',       'Original — use rest_type_enc or OHE for models'),
    ('cuisines',                     'object',  'Multi-value', 'Recommendation',  'Original — use cuisine_count for ML'),
    ('approx_cost(for two people)',  'float64', 'Numerical',   'All models',      'Median-imputed in Phase 2 — ready for modelling'),
    ('listed_in(type)',              'object',  'Categorical', 'Reference',       'Original — use listed_intype_enc for models'),
    ('listed_in(city)',              'object',  'Categorical', 'Reference',       'Original — use listed_incity_enc or OHE for models'),
    ('reviews_list',                 'object',  'NLP',         'NLP pipeline',    'Preserved — TF-IDF/embeddings in later notebooks'),
    ('dish_liked',                   'object',  'NLP',         'NLP pipeline',    'Preserved — high null expected'),
    ('menu_item',                    'object',  'RAG',         'RAG pipeline',    'Preserved — chunk and index in RAG notebook'),
    # Engineered columns
    ('rating_category',              'category','Target',      'Classification',  'Binned: Poor/Average/Good/Excellent'),
    ('rating_category_enc',          'float64', 'Target',      'Classification',  '0=Poor 1=Average 2=Good 3=Excellent'),
    ('cost_category',                'category','Engineered',  'Reference',       'Budget/Mid-Range/Premium/Luxury'),
    ('cost_category_enc',            'float64', 'Engineered',  'All models',      '0=Budget 1=Mid-Range 2=Premium 3=Luxury'),
    ('cuisine_count',                'int64',   'Engineered',  'All models',      'Count of distinct cuisines — 0 if Unknown'),
    ('dish_count',                   'int64',   'Engineered',  'All models',      'Count of popular dishes in dish_liked'),
    ('review_count',                 'int64',   'Engineered',  'All models',      'Count of individual reviews in reviews_list'),
    ('votes_log',                    'float64', 'Engineered',  'All models',      'log1p(votes) — reduces right skew'),
    ('rpi',                          'float64', 'Engineered',  'All models',      'rate × log(1+votes) — undefined for unrated rows'),
    ('online_order_enc',             'int64',   'Encoded',     'All models',      'Yes=1 No=0'),
    ('book_table_enc',               'int64',   'Encoded',     'All models',      'Yes=1 No=0'),
    ('location_enc',                 'int64',   'Encoded',     'Tree + LightGBM', 'LabelEncoder — 93+ classes'),
    ('rest_type_enc',                'int64',   'Encoded',     'Tree + LightGBM', 'LabelEncoder — 93+ classes'),
    ('listed_intype_enc',            'int64',   'Encoded',     'All models',      'LabelEncoder — 7 classes'),
    ('listed_incity_enc',            'int64',   'Encoded',     'Tree + LightGBM', 'LabelEncoder — 30 classes'),
]

print('  DATA DICTIONARY — Feature-Engineered Dataset')
print('  ' + '=' * 120)
print(f'  {"Column":<35} {"Dtype":<10} {"Category":<14} {"Used In":<20} {"Notes"}')
print('  ' + '-' * 120)
for row in data_dict:
    col, dtype, cat, used, notes = row
    print(f'  {col:<35} {dtype:<10} {cat:<14} {used:<20} {notes}')
print('  ' + '=' * 120)

  DATA DICTIONARY — Feature-Engineered Dataset
  Column                              Dtype      Category       Used In              Notes
  ------------------------------------------------------------------------------------------------------------------------
  name                                object     Identifier     Reference            Restaurant name — not used as feature
  rate                                float64    Target         Regression           Continuous target — null rows excluded at train time
  votes                               int64      Numerical      Tree + LightGBM      Raw vote count — use votes_log for models
  online_order                        object     Binary         Reference            Original — use online_order_enc for models
  book_table                          object     Binary         Reference            Original — use book_table_enc for models
  location                            object     Categorical    Reference            Original — u

## 14 · Export Datasets

In [28]:
# ── Full engineered dataset (all rows — for Flask, recommendation, RAG) ──
FULL_ENG_PATH = OUTPUT_DIR / 'zomato_engineered_full.csv'
df.to_csv(FULL_ENG_PATH, index=False)
print(f'Full engineered dataset  → {FULL_ENG_PATH.name}  |  {df.shape}')

# ── Supervised-only dataset (rows with valid rate) ────────────────────────
SUPERVISED_PATH = OUTPUT_DIR / 'zomato_supervised.csv'
df_supervised.to_csv(SUPERVISED_PATH, index=False)
print(f'Supervised dataset       → {SUPERVISED_PATH.name}  |  {df_supervised.shape}')

# ── Tree model splits ─────────────────────────────────────────────────────
X_train_tree.to_csv(OUTPUT_DIR / 'X_train_tree.csv',  index=False)
X_test_tree.to_csv(OUTPUT_DIR  / 'X_test_tree.csv',   index=False)
y_train_reg.to_csv(OUTPUT_DIR  / 'y_train_reg.csv',   index=False, header=True)
y_test_reg.to_csv(OUTPUT_DIR   / 'y_test_reg.csv',    index=False, header=True)
y_train_clf.to_csv(OUTPUT_DIR  / 'y_train_clf.csv',   index=False, header=True)
y_test_clf.to_csv(OUTPUT_DIR   / 'y_test_clf.csv',    index=False, header=True)

print(f'X_train_tree             → X_train_tree.csv    |  {X_train_tree.shape}')
print(f'X_test_tree              → X_test_tree.csv     |  {X_test_tree.shape}')
print(f'y_train_reg, y_test_reg  → saved')
print(f'y_train_clf, y_test_clf  → saved')

# ── SVM splits ────────────────────────────────────────────────────────────
X_train_svm.to_csv(OUTPUT_DIR / 'X_train_svm_scaled.csv', index=False)
X_test_svm.to_csv(OUTPUT_DIR  / 'X_test_svm_scaled.csv',  index=False)
y_train_svm.to_csv(OUTPUT_DIR / 'y_train_svm.csv',        index=False, header=True)
y_test_svm.to_csv(OUTPUT_DIR  / 'y_test_svm.csv',         index=False, header=True)

print(f'X_train_svm_scaled       → X_train_svm_scaled.csv  |  {X_train_svm.shape}')
print(f'X_test_svm_scaled        → X_test_svm_scaled.csv   |  {X_test_svm.shape}')
print(f'y_train_svm, y_test_svm  → saved')

# ── NLP dataset ───────────────────────────────────────────────────────────
NLP_PATH = OUTPUT_DIR / 'zomato_nlp_dataset.csv'
df_nlp.to_csv(NLP_PATH, index=False)
print(f'NLP dataset              → {NLP_PATH.name}  |  {df_nlp.shape}')

print('\n✅ All datasets exported successfully.')
print()
print('Exported files summary:')
print('  zomato_engineered_full.csv   — complete engineered dataset (all rows)')
print('  zomato_supervised.csv        — rows with valid rate (for regression + classification)')
print('  X_train_tree.csv             — tree model training features')
print('  X_test_tree.csv              — tree model test features')
print('  X_train_svm_scaled.csv       — SVM training features (StandardScaler applied)')
print('  X_test_svm_scaled.csv        — SVM test features (StandardScaler applied)')
print('  zomato_nlp_dataset.csv       — text + context for NLP and RAG pipelines')

Full engineered dataset  → zomato_engineered_full.csv  |  (51717, 29)
Supervised dataset       → zomato_supervised.csv  |  (41665, 29)
X_train_tree             → X_train_tree.csv    |  (33332, 14)
X_test_tree              → X_test_tree.csv     |  (8333, 14)
y_train_reg, y_test_reg  → saved
y_train_clf, y_test_clf  → saved
X_train_svm_scaled       → X_train_svm_scaled.csv  |  (33332, 225)
X_test_svm_scaled        → X_test_svm_scaled.csv   |  (8333, 225)
y_train_svm, y_test_svm  → saved
NLP dataset              → zomato_nlp_dataset.csv  |  (51717, 11)

✅ All datasets exported successfully.

Exported files summary:
  zomato_engineered_full.csv   — complete engineered dataset (all rows)
  zomato_supervised.csv        — rows with valid rate (for regression + classification)
  X_train_tree.csv             — tree model training features
  X_test_tree.csv              — tree model test features
  X_train_svm_scaled.csv       — SVM training features (StandardScaler applied)
  X_test_svm_scaled.

## 15 · Pipeline Flow Summary

This notebook is the **branching point** of the entire Zomato ML pipeline. Every downstream notebook reads from one of the outputs created here.

```
01_Data_Profiling.ipynb
        │
        ▼
02_Data_Cleaning.ipynb
  (clean + impute + preserve all rows including unrated)
        │
        ▼
03_Feature_Engineering.ipynb   ◄── YOU ARE HERE
  (engineer features · encode · scale · split)
        │
        ├─── zomato_engineered_full.csv
        │         └─► Flask Application
        │         └─► Recommendation Engine
        │
        ├─── zomato_supervised.csv
        │         └─► (internal — used to generate splits below)
        │
        ├─── X_train_tree.csv / X_test_tree.csv
        │         └─► 04_DecisionTree.ipynb
        │         └─► 05_LightGBM.ipynb
        │
        ├─── X_train_svm_scaled.csv / X_test_svm_scaled.csv
        │         └─► 06_SVM.ipynb
        │
        └─── zomato_nlp_dataset.csv
                  └─► 07_NLP_Analysis.ipynb
                  └─► 08_RAG_Pipeline.ipynb
```

**Key design decisions made in this notebook:**
- Cleaning and feature engineering are kept as separate, explicit phases
- Unrated restaurants are preserved until Section 10 — the first and only exclusion point
- Encoding strategy is model-specific (Label for trees, OHE for SVM)
- StandardScaler is fitted on training data only — no information from test rows leaks into the model
- RPI is intentionally undefined for unrated restaurants — this is correct behaviour, not a data error